# Testing MAS for whole data management

This journal proposes a variant of implementation and testing of a general multi-agent system that executes user requests using separate workflows, which are presented as separate tools. 

In this implementation, the workflow for generating metadata became a tool for creating datasets, the workflow for adding data became a tool for adding data, and the workflow for receiving data became the corresponding tool.

## 0. Initial imports and variables

In [ ]:
import sys

sys.path.append('./../src')

In [ ]:
import os
import yaml

MODEL_NAME = 'mistral-large-latest'
PROVIDER = 'mistralai'

CREATING_DS_PROMPT_PART = 'creating_md/chrono_creating_dataset_sp.yaml'
ADDING_PROMPT_PART = 'adding_data_to_dataset/adding_data_sp.yaml'
GETTING_PROMPT_PART = 'getting_data/getting_data_sp.yaml'
ORCHESTRATOR_PROMPT_PART = 'data_manager_orchestrator_sp.yaml'

with open(os.path.join('../src/prompts_templates/', CREATING_DS_PROMPT_PART)) as stream:
    CREATING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', ADDING_PROMPT_PART)) as stream:
    ADDING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', GETTING_PROMPT_PART)) as stream:
    GETTING_SP = yaml.safe_load(stream)['system_prompt']
with open(os.path.join('../src/prompts_templates/', ORCHESTRATOR_PROMPT_PART)) as stream:
    ORCHESTRATOR_SP = yaml.safe_load(stream)['system_prompt']

## 1. Building elements of MAS

### 1.0 Initing datalake from readed instance 

To demonstrate the work, as in getting data demonstration, it is recommended to first run the results of the data addition demonstration - during the work of the journal, a test data lake will be created, which will be filled with data with metadata values.

In [ ]:
from data_management.local_datalake_management import LocalDataLake
readed_datalake = LocalDataLake.open("./add_testing_storage")
readed_datalake.get_datalake_info()

### 1.1 Initing tool for generating dataset

In [ ]:
from langchain_core.tools import StructuredTool

from scidatamas.dataset_creating_workflow import MDGenerationFlow, MDGenerationState

class GeneratingDatasetTool():
    def __init__(self, system_prompt: str, datalake:LocalDataLake, model: str, provider: str):
        self.__creating_datasets_workflow = MDGenerationFlow(system_prompt=system_prompt, model=model, provider=provider, is_auto=True)
        self.__wf = self.__creating_datasets_workflow.get_workflow()
        self.__wf = self.__wf.compile()
        self.__datalake = datalake
        self.tool = StructuredTool.from_function(func=self.__generate_dataset_tool, parse_docstring=True)
        return
    
    def __generate_dataset_tool(self, user_task: str) -> str:
        """
        Generating dataset in datalake agentic workflow.

        Args:
            user_task (str): User's task which describes requirements about created dataset

        Returns:
            str: log information from workflow
        """
        working_state = MDGenerationState()
        working_state['users_task'] = user_task
        working_state = self.__wf.invoke(working_state)
        
        generated_schema = working_state['generation']
        self.__datalake.create_dataset(name=generated_schema.short_name, description=generated_schema.description, dataset_meta_schema=generated_schema.schema)

        metadata_str_descr = "".join([f'\n - {k}: {v["descr"]} ({v["type"]})' for k, v in generated_schema.schema.items()])
        tool_answer = (
            "Generated next dataset:\n\n"
            + f"Name: {generated_schema.short_name}\n\n"
            + f"Description: {generated_schema.short_name}\n\n"
            + f"Schema (described like rows of 'field_name: description (type)'): \n"
            + f'{metadata_str_descr}'
        )
        return tool_answer

### 1.2 Initing tool for adding data

In [ ]:
from scidatamas.add_data_workflow import AddDataToDatasetFlow, AddDataWorkflowState

class AddingDataToDataLakeTool():
    def __init__(self, system_prompt: str, datalake:LocalDataLake, model: str, provider: str):
        self.__adding_data_workflow = AddDataToDatasetFlow(system_prompt=system_prompt, datalake=datalake, model=model, provider=provider, is_auto=True)
        self.__wf = self.__adding_data_workflow.get_workflow()
        self.__wf = self.__wf.compile()

        self.tool = StructuredTool.from_function(func=self.__add_data_tool, parse_docstring=True)
        
        return
    
    def __add_data_tool(self, user_task: str) -> str:
        """
        Adding data in datalake agentic workflow.

        Args:
            user_task (str): User's task which describes adding data task for workflow

        Returns:
            str: log information from workflow's running
        """
        working_state = AddDataWorkflowState()
        working_state['users_task'] = user_task
        working_state = self.__wf.invoke(working_state)
        
        added_data = working_state['data_to_add']
        dataset_destination = working_state['choosen_dataset']
        
        filled_meta = working_state['currently_filled_schema']
        origin_meta = working_state['origin_dataset_meta_schema']
        skipped_fields = list(set(origin_meta.keys()) - set(filled_meta.keys()))
        metadata_str_descr = "".join([f'\n - {k}: {v}' for k, v in filled_meta.items()])
        
        tool_answer = (
            "The data adding workflow result:\n\n"
            + f"Added data: {added_data}\n\n"
            + f"Dataset destination (where data was added and saved): {dataset_destination}\n\n"
            + f"Filled properties of schema: {metadata_str_descr}"
        )

        if len(skipped_fields) > 0:
            skipepd_fields_str = "".join([f'\n - {sk}' for sk in skipped_fields])
            tool_answer += f'\n\nMissed fields of metadata: {skipepd_fields_str}'

        return tool_answer

### 1.3 Initing tool for getting data

In [ ]:
import pandas as pd 

from scidatamas.data_retrieval_workflow import DataRetrievingFlow, RetrievingDatalakeDataState

class RetrievingDataTool():
    def __init__(self, system_prompt: str, datalake:LocalDataLake, model: str, provider: str, max_iters: int):
        self.__getting_data_workflow = DataRetrievingFlow(system_prompt=system_prompt, datalake=datalake, model=model, provider=provider, max_iterations=max_iters)
        self.__wf = self.__getting_data_workflow.get_workflow()
        self.__wf = self.__wf.compile()

        self.tool = StructuredTool.from_function(func=self.__retrieving_data_tool, parse_docstring=True)
        
        return
    
    def __retrieving_data_tool(self, user_task: str) -> str:
        """
        Retrieving data from datalake agentic workflow.

        Args:
            user_task (str): User's task which describes which data he wants to get

        Returns:
            str: log information from workflow's running
        """
        working_state = RetrievingDatalakeDataState()
        working_state['users_task'] = user_task
        working_state = self.__wf.invoke(working_state)
        
        retrieving_result = working_state['data_retrieving_result']
        error_log = working_state['error_log']
        dataset_source = working_state['dataset_source_name']
        df = working_state['data_retrieving_result']
     
        tool_answer = ""

        if isinstance(retrieving_result, pd.DataFrame) == True:
            tool_answer = (
                "Succesfully extracted data.\n\n"
                + f"The descripion of data is:\n{str(df.iloc[0:min(5, df.shape[0])].to_string())}\n\n"
                + f"The query for dataset should looks like this:\n```{working_state['sql_generation']}```"
            )
        elif retrieving_result == None and error_log != '':
            tool_answer = f"During data extracton from datalake an error was observed:\n\"{error_log}\"."
        elif retrieving_result == None and error_log == '' and dataset_source == None:
            tool_answer = "Can't extract data because relevant dataset wasn't found (if you can - describe or call the dataset's name)."
        else:
            tool_answer = "Can't extract data due to unknown error (maybe there is just no data)."


        return tool_answer

### 1.4 Initing final MAS

It is important to mention, that below code contains different tools for working wirh data lake. You can use different tools also, but it is important to descrive them into system prompt. 

In [ ]:
from scidatamas.dataset_manager import DatalakeManagerAgent, OrchestratorState

generating_dataset_tool = GeneratingDatasetTool(CREATING_SP, readed_datalake, MODEL_NAME, PROVIDER)
adding_data_tool = AddingDataToDataLakeTool(ADDING_SP, readed_datalake, MODEL_NAME, PROVIDER)
getting_data_tool = RetrievingDataTool(GETTING_SP, readed_datalake, MODEL_NAME, PROVIDER, 5)

mas_tools = [generating_dataset_tool.tool, adding_data_tool.tool, getting_data_tool.tool]

dataset_manager_mas = DatalakeManagerAgent(system_prompt=ORCHESTRATOR_SP, workflows_tools=mas_tools, model=MODEL_NAME, provider=PROVIDER)
dm_mas_wf = dataset_manager_mas.get_workflow()
dm_mas_wf = dm_mas_wf.compile()

def call_mas(user_task: str):
    working_state = OrchestratorState()
    working_state['users_task'] = user_task
    working_state['messages'] = [('user', user_task)]
    working_state = dm_mas_wf.invoke(working_state)
    return working_state

## 2. Demonstration MAS working

### 2.1 Creating dataset

In [ ]:
USER_CREARE_DS_TASK = """Hi!
I want to create a metadata schema for data that evaluates the behavior of mice in a circular arena. The dataset contains pairs of two videos: one video is obtained from a camera that shoots a mouse in a circular arena, and the second video captures the activity of neurons in the mouse's hippocampus, assessing which neurons are activated and which are not during different actions.
Such a dataset is useful for analyzing the eating of healthy and neurodegenerative disease-affected mice, comparing brain activity and their behavior. Please help me."""

res = call_mas(USER_CREARE_DS_TASK)
res['result']

### 2.2 Getting data

In [ ]:
USER_GET_DATA_TASK = "Hi! I need some data captured by fluorescent microscope, where pixel sizes along OX and OY axies are more then 1024 an less then 3096. Can you get it from me?"

res = call_mas(USER_GET_DATA_TASK)
res['result']

### 2.3 Adding data

In [ ]:
USER_ADD_DATA_TASK = "Hi! Add neuron images from the directory './raw_testing_material/imgs/' to my dataset fluorescent images dataset. To describe images use metadata specified in the file './raw_testing_material/imgs_meta.xml'"

res = call_mas(USER_ADD_DATA_TASK)
res['result']